# Phase 1 Experiment: Phishing Email Detection

This notebook mirrors the reproducible project pipeline: load config, inspect raw and processed data, review the configured TF-IDF setup, train baseline models, and review evaluation metrics. The production scripts remain the source of truth; notebook cells call those scripts/functions instead of duplicating pipeline logic.

In [ ]:
import joblib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, PrecisionRecallDisplay

from mlops_crew.config import CONFIG_PATH, load_project_config, resolve_project_path
from mlops_crew.data import LABEL_COLUMN, TEXT_COLUMN
from mlops_crew.data.make_dataset import process_data
from mlops_crew.train_model import train

config = load_project_config(CONFIG_PATH)
config

## Raw Data Check

Phase 1 intentionally uses a reproducible 60% sample of `phishing_email.csv`. The full raw file and sampled file both use the required schema: `text_combined,label`.

In [ ]:
raw_path = resolve_project_path(config["data"]["raw_dir"]) / config["data"]["raw_file"]
raw = pd.read_csv(raw_path)
print("Configured sample fraction:", config["data"]["sample"]["fraction"])
print(raw.shape)
display(raw.head())
display(raw[LABEL_COLUMN].value_counts(normalize=True).rename("class_ratio"))

## Reproducible Data Prep

This runs the same sample, clean, split, and validation logic used by `make data`. DVC runs those stages separately for reproducibility.

In [ ]:
process_data(CONFIG_PATH)

processed_dir = resolve_project_path(config["data"]["processed_dir"])
train_df = pd.read_csv(processed_dir / config["data"]["train_file"])
val_df = pd.read_csv(processed_dir / config["data"]["val_file"])
test_df = pd.read_csv(processed_dir / config["data"]["test_file"])

split_summary = (
    pd.DataFrame(
        {
            "train": train_df[LABEL_COLUMN].value_counts(),
            "val": val_df[LABEL_COLUMN].value_counts(),
            "test": test_df[LABEL_COLUMN].value_counts(),
        }
    )
    .fillna(0)
    .astype(int)
)
display(split_summary)
display(train_df.head())

## TF-IDF Vocabulary Spot-Check

TF-IDF is part of the saved sklearn pipeline. For a quick pre-training sanity check, we fit the configured vectorizer on the training split only and inspect a few vocabulary terms.

In [ ]:
from mlops_crew.models.text_classifiers import build_tfidf_vectorizer

vectorizer = build_tfidf_vectorizer(config).fit(train_df[TEXT_COLUMN])
print("Vocabulary size:", len(vectorizer.vocabulary_))
sample_terms = pd.Series(vectorizer.get_feature_names_out()).sample(
    15, random_state=config["project"]["seed"]
)
display(sample_terms.to_frame("sample_terms"))

## Train And Evaluate

The training script fits the configured baseline models and saves full sklearn pipeline artifacts under `models/`.

In [ ]:
results = train(config)
metrics_dir = resolve_project_path(config["reports"]["metrics_dir"])
comparison = pd.read_csv(metrics_dir / "model_comparison.csv")
display(comparison.T)
results["best_model"]

## Best Model Diagnostics

For phishing detection, false negatives are especially important because they represent phishing emails classified as safe.

In [ ]:
model_dir = resolve_project_path(config["modeling"]["output_dir"])
best_model = joblib.load(model_dir / "best_model.joblib")

test_predictions = best_model.predict(test_df[TEXT_COLUMN])
test_scores = best_model.predict_proba(test_df[TEXT_COLUMN])[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ConfusionMatrixDisplay.from_predictions(
    test_df[LABEL_COLUMN],
    test_predictions,
    ax=axes[0],
)
axes[0].set_title("Test Confusion Matrix")
PrecisionRecallDisplay.from_predictions(
    test_df[LABEL_COLUMN],
    test_scores,
    ax=axes[1],
)
axes[1].set_title("Test Precision-Recall Curve")
plt.tight_layout()

In [ ]:
errors = test_df.assign(prediction=test_predictions, score=test_scores)
false_negatives = errors[(errors[LABEL_COLUMN] == 1) & (errors["prediction"] == 0)]
false_positives = errors[(errors[LABEL_COLUMN] == 0) & (errors["prediction"] == 1)]

print("False negatives:", len(false_negatives))
print("False positives:", len(false_positives))
display(false_negatives[[TEXT_COLUMN, LABEL_COLUMN, "prediction", "score"]].head())
display(false_positives[[TEXT_COLUMN, LABEL_COLUMN, "prediction", "score"]].head())